In [ ]:
!pip install -q "flash-linear-attention[cuda]==0.5.2"
!pip install -q "transformers==5.13.1" datasets accelerate evaluate

!rm -rf /kaggle/working/whisper-kda-asr

!git clone -q https://github.com/SummerSchoolSPBU2026/whisper-kda-asr.git /kaggle/working/whisper-kda-asr

!git -C /kaggle/working/whisper-kda-asr checkout -q 4b0bf91345f911219401bbaafda597f33dfb9183

print("Репозиторий загружен")

Репозиторий загружен


In [ ]:
import os
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import math
import time
import torch
import importlib.metadata

REPO_PATH = "/kaggle/working/whisper-kda-asr"
MODEL_PATH = os.path.join(REPO_PATH, "model.py")

print("Репозиторий найден:", os.path.isdir(REPO_PATH))
print("model.py найден:", os.path.isfile(MODEL_PATH))

assert os.path.isdir(REPO_PATH), (
    "Папка репозитория не найдена. "
    "Сначала запусти предыдущую ячейку с git clone."
)

assert os.path.isfile(MODEL_PATH), (
    f"Файл model.py не найден по пути: {MODEL_PATH}"
)

# Добавляем корень репозитория, потому что model.py лежит именно там
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Удаляем возможный старый импорт model из памяти
sys.modules.pop("model", None)

from model import KDACrossAttentionDecoder
from datasets import load_dataset

from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

from fla.models import KDAConfig

device = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

print("torch:", torch.__version__)
print(
    "flash-linear-attention:",
    importlib.metadata.version("flash-linear-attention")
)
print(
    "transformers:",
    importlib.metadata.version("transformers")
)
print(
    "model class:",
    KDACrossAttentionDecoder.__name__
)

Репозиторий найден: True
model.py найден: True
device: cuda
GPU: Tesla T4
GPU count: 1
torch: 2.10.0+cu128
flash-linear-attention: 0.5.2
transformers: 5.13.1
model class: KDACrossAttentionDecoder


In [ ]:
print("GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))

GPU count: 1
GPU: Tesla T4


In [ ]:
# Блок 1 — текстовый корпус для языкового претрейна

raw_texts = load_dataset(
    "google/fleurs",
    "ru_ru",
    split="train[:1000]"
)

text_dataset = raw_texts.select_columns(["transcription"])
text_dataset = text_dataset.rename_column("transcription", "text")

text_dataset = text_dataset.filter(
    lambda x: (
        x["text"] is not None
        and len(x["text"].strip()) > 0
    )
)

split_dataset = text_dataset.train_test_split(
    test_size=0.1,
    seed=42
)

print(split_dataset)
print(split_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 900
    })
    test: Dataset({
        features: ['text'],
        num_rows: 100
    })
})
{'text': 'ранее тем же днем харке тренировался во время предсезонной подготовки в коверчано в италии он оставался в отеле команды перед запланированным на воскресенье матчем против болоньи'}


In [ ]:
# Блок 2 — финальный BPE-токенизатор

import glob

from tokenizers import Tokenizer
from tokenizers.processors import TemplateProcessing
from transformers import PreTrainedTokenizerFast

# Kaggle сам кладёт загруженные Input-файлы в /kaggle/input
tokenizer_paths = glob.glob(
    "/kaggle/input/**/bpe_tokenizer.json",
    recursive=True
)

assert tokenizer_paths, "bpe_tokenizer.json не найден в Kaggle Input"

TOKENIZER_PATH = tokenizer_paths[0]
print("Tokenizer path:", TOKENIZER_PATH)

# Загружаем BPE-токенизатор из JSON
tokenizer_backend = Tokenizer.from_file(TOKENIZER_PATH)

bos_id = tokenizer_backend.token_to_id("<s>")
eos_id = tokenizer_backend.token_to_id("</s>")

# В исходном JSON post_processor отсутствует,
# поэтому явно добавляем <s> в начало и </s> в конец текста
tokenizer_backend.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    special_tokens=[
        ("<s>", bos_id),
        ("</s>", eos_id),
    ],
)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_backend,
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<s>",
    eos_token="</s>",
)

MAX_LEN = 128

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
        add_special_tokens=True,
    )

tokenized_dataset = split_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

print("vocab size:", len(tokenizer))
print("unk:", tokenizer.unk_token_id)
print("pad:", tokenizer.pad_token_id)
print("bos:", tokenizer.bos_token_id)
print("eos:", tokenizer.eos_token_id)

test_ids = tokenizer("проверка токенизатора")["input_ids"]
print("test input_ids:", test_ids)
print("first token:", test_ids[0])
print("last token:", test_ids[-1])

Tokenizer path: /kaggle/input/datasets/kakashka123/bpe-tokenizer-final/bpe_tokenizer.json
vocab size: 6390
unk: 0
pad: 1
bos: 2
eos: 3
test input_ids: [2, 91, 233, 49, 44, 193, 53, 59, 3377, 3]
first token: 2
last token: 3


In [ ]:
# Блок 3 — текстовый collator

base_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

def data_collator(features):
    batch = base_data_collator(features)

    # Командный декодер ожидает маску под этим названием
    batch["decoder_attention_mask"] = batch["attention_mask"]

    return batch

batch = data_collator([
    tokenized_dataset["train"][0],
    tokenized_dataset["train"][1],
])

print(batch.keys())
print("input_ids:", batch["input_ids"].shape)
print("labels:", batch["labels"].shape)
print(
    "decoder_attention_mask:",
    batch["decoder_attention_mask"].shape,
)

KeysView({'input_ids': tensor([[   2,   52,  218,  463,   77, 3259,  202,   25,  193,  211,   53,   54,
          691,   55,  941,  446,   78,   16,   82,  194,  191, 3096,   88,   11,
           50,  233,   90,   51,   11,   17, 1549,   17,   82,  200,  691,   11,
         1267,   68,   50,  596,  209,  447, 2397,   53, 4596,  990,   42, 6210,
           78, 1229,   75,   27,  248, 1361,   98,   60,  124,   17,    3],
        [   2,   67, 2348, 1276, 3035,    0,    0,    0,  616,   91,  731, 2935,
           17,  146,   81,  351,  117, 3429, 3760,   10,   21,   12,   51,  580,
          465,  116, 1444,  422, 4004, 2162, 1528,  809,  463,  142,   97, 5649,
          758,   72,   82, 3179,  794,   76,  988,  392,   11,   26,  149,    3,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [ ]:
# Блок 4 — KDA decoder в text-only режиме

config = KDAConfig(
    vocab_size=len(tokenizer),

    hidden_size=384,
    num_hidden_layers=4,
    num_heads=6,
    head_dim=64,
    num_v_heads=6,
    expand_v=1.0,
    intermediate_size=1024,

    attn_mode="chunk",
    attn=None,
    use_short_conv=True,
    conv_size=4,
    norm_eps=1e-6,
    attnres_block_size=None,

    tie_word_embeddings=True,
    use_cache=False,
    fuse_norm=True,
    fuse_swiglu=True,
    fuse_cross_entropy=False,

    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,

    is_encoder_decoder=False,
)

kda_decoder = KDACrossAttentionDecoder(config).to(device)

# Cross-attention существует в модели, но при языковом
# претрейне не используется и не обучается.
for layer in kda_decoder.model.layers:
    layer.cross_attn.requires_grad_(False)
    layer.cross_attn_norm.requires_grad_(False)

total_params = sum(
    parameter.numel()
    for parameter in kda_decoder.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in kda_decoder.parameters()
    if parameter.requires_grad
)

print("model class:", type(kda_decoder).__name__)
print("total params:", total_params)
print("trainable params:", trainable_params)
print(kda_decoder)

model class: KDACrossAttentionDecoder
total params: 12326296
trainable params: 9959320
KDACrossAttentionDecoder(
  (model): KDACrossAttentionBackbone(
    (embeddings): Embedding(6390, 384, padding_idx=1)
    (layers): ModuleList(
      (0-3): 4 x KDACrossAttentionBlock(
        (attn_norm): RMSNorm(384, eps=1e-06)
        (attn): KimiDeltaAttention(
          (q_proj): Linear(in_features=384, out_features=384, bias=False)
          (k_proj): Linear(in_features=384, out_features=384, bias=False)
          (v_proj): Linear(in_features=384, out_features=384, bias=False)
          (q_conv1d): ShortConvolution(384, 384, kernel_size=(4,), stride=(1,), padding=(3,), groups=384, bias=False, activation=silu, backend=triton)
          (k_conv1d): ShortConvolution(384, 384, kernel_size=(4,), stride=(1,), padding=(3,), groups=384, bias=False, activation=silu, backend=triton)
          (v_conv1d): ShortConvolution(384, 384, kernel_size=(4,), stride=(1,), padding=(3,), groups=384, bias=False, activ

In [ ]:
# Проверка text-only режима на одном batch

smoke_batch = {
    key: value.to(device)
    for key, value in batch.items()
}

kda_decoder.eval()

with torch.no_grad():
    smoke_outputs = kda_decoder(
        input_ids=smoke_batch["input_ids"],
        decoder_attention_mask=smoke_batch[
            "decoder_attention_mask"
        ],
        labels=smoke_batch["labels"],
        encoder_hidden_states=None,
        use_cache=False,
    )

print("logits shape:", smoke_outputs.logits.shape)
print("loss:", smoke_outputs.loss.item())

assert smoke_outputs.loss is not None
assert torch.isfinite(smoke_outputs.loss)

print("Text-only forward: OK")

kda_decoder.train()

logits shape: torch.Size([2, 59, 6390])
loss: 8.863922119140625
Text-only forward: OK


KDACrossAttentionDecoder(
  (model): KDACrossAttentionBackbone(
    (embeddings): Embedding(6390, 384, padding_idx=1)
    (layers): ModuleList(
      (0-3): 4 x KDACrossAttentionBlock(
        (attn_norm): RMSNorm(384, eps=1e-06)
        (attn): KimiDeltaAttention(
          (q_proj): Linear(in_features=384, out_features=384, bias=False)
          (k_proj): Linear(in_features=384, out_features=384, bias=False)
          (v_proj): Linear(in_features=384, out_features=384, bias=False)
          (q_conv1d): ShortConvolution(384, 384, kernel_size=(4,), stride=(1,), padding=(3,), groups=384, bias=False, activation=silu, backend=triton)
          (k_conv1d): ShortConvolution(384, 384, kernel_size=(4,), stride=(1,), padding=(3,), groups=384, bias=False, activation=silu, backend=triton)
          (v_conv1d): ShortConvolution(384, 384, kernel_size=(4,), stride=(1,), padding=(3,), groups=384, bias=False, activation=silu, backend=triton)
          (f_proj): Sequential(
            (0): Linear(in_

In [ ]:
print(len(tokenizer))

6390


In [ ]:
# Блок 5 — настройки финального претрейна KDA decoder
training_args = TrainingArguments(
    output_dir="/kaggle/working/decoder_pretrain_team_kda",

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=3e-4,
    max_steps=200,

    fp16=torch.cuda.is_available(),

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=25,

    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to=[]
)

trainer = Trainer(
    model=kda_decoder,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

In [ ]:
# Блок 6 — измеряем качество модели ДО обучения.

# Trainer передаёт input_ids, labels и decoder_attention_mask
# непосредственно в KDACrossAttentionDecoder.
# encoder_hidden_states=None, поэтому cross-attention не выполняется.
eval_before = trainer.evaluate()

loss_before = eval_before["eval_loss"]

# Переводим loss в perplexity (чем меньше, тем лучше)
ppl_before = math.exp(loss_before)

print("eval_loss before:", loss_before)
print("perplexity before:", ppl_before)

Training Loss,Validation Loss,Step
No log,8.811097,0


eval_loss before: 8.811097145080566
perplexity before: 6708.27518214237


In [ ]:
# Запоминаем текущее время перед началом обучения.
start_time = time.time()

# Запускаем языковой претрейн на токенизированных текстах.
# Whisper encoder, аудио и cross-attention здесь не используются.
train_result = trainer.train()

train_time = time.time() - start_time

print("train done")
print("time sec:", round(train_time, 2))
print(train_result)

# После обучения снова проверяем модель на validation-данных.
eval_after = trainer.evaluate()

loss_after = eval_after["eval_loss"]
ppl_after = math.exp(loss_after)

print("eval_loss after:", loss_after)
print("perplexity after:", ppl_after)

Step,Training Loss,Validation Loss
25,67.682959,7.991044
50,53.658667,6.590798
75,48.566251,6.388367
100,46.102197,6.200690
125,39.308115,6.031194
150,38.227661,5.850664
175,38.182541,5.770228
200,34.563962,5.748242


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


train done
time sec: 114.9
TrainOutput(global_step=200, training_loss=46.2850830078125, metrics={'train_runtime': 114.5289, 'train_samples_per_second': 27.941, 'train_steps_per_second': 1.746, 'total_flos': 9750827376192.0, 'train_loss': 46.2850830078125, 'epoch': 3.5155555555555553})


Training Loss,Validation Loss,Step
34.563962,5.748242,200


eval_loss after: 5.748242378234863
perplexity after: 313.63891696284855


In [ ]:
# Блок 7 — сохраняем итоговый checkpoint и метрики

import os
import json

save_path = (
    "/kaggle/working/"
    "decoder_pretrain_team_kda/final_checkpoint"
)

os.makedirs(save_path, exist_ok=True)

# Сохраняем модель, которая находится внутри Trainer.
# При load_best_model_at_end=True это лучшая версия
# по validation loss.
trainer.save_model(save_path)

# Сохраняем токенизатор Маргариты
tokenizer.save_pretrained(save_path)

# Сохраняем результаты эксперимента
metrics = {
    "vocab_size": len(tokenizer),
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "max_steps": training_args.max_steps,
    "effective_batch_size": (
        training_args.per_device_train_batch_size
        * training_args.gradient_accumulation_steps
    ),
    "eval_loss_before": float(loss_before),
    "perplexity_before": float(ppl_before),
    "eval_loss_after": float(loss_after),
    "perplexity_after": float(ppl_after),
    "training_time_seconds": float(train_time),
    "best_eval_loss": (
        float(trainer.state.best_metric)
        if trainer.state.best_metric is not None
        else None
    ),
    "best_checkpoint": trainer.state.best_model_checkpoint,
}

with open(
    os.path.join(save_path, "pretrain_metrics.json"),
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("saved to:", save_path)
print("saved files:", os.listdir(save_path))
print("metrics:", metrics)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to: /kaggle/working/decoder_pretrain_team_kda/final_checkpoint
saved files: ['tokenizer_config.json', 'model.safetensors', 'training_args.bin', 'pretrain_metrics.json', 'config.json', 'generation_config.json', 'tokenizer.json']
metrics: {'vocab_size': 6390, 'total_parameters': 12326296, 'trainable_parameters': 9959320, 'max_steps': 200, 'effective_batch_size': 16, 'eval_loss_before': 8.811097145080566, 'perplexity_before': 6708.27518214237, 'eval_loss_after': 5.748242378234863, 'perplexity_after': 313.63891696284855, 'training_time_seconds': 114.90364861488342, 'best_eval_loss': 5.748242378234863, 'best_checkpoint': '/kaggle/working/decoder_pretrain_team_kda/checkpoint-200'}


In [ ]:
!cd /kaggle/working && zip -r \
kda_team_decoder_final_checkpoint.zip \
decoder_pretrain_team_kda/final_checkpoint

  adding: decoder_pretrain_team_kda/final_checkpoint/ (stored 0%)
  adding: decoder_pretrain_team_kda/final_checkpoint/tokenizer_config.json (deflated 43%)
  adding: decoder_pretrain_team_kda/final_checkpoint/model.safetensors (deflated 8%)
  adding: decoder_pretrain_team_kda/final_checkpoint/training_args.bin (deflated 54%)
  adding: decoder_pretrain_team_kda/final_checkpoint/pretrain_metrics.json (deflated 44%)
  adding: decoder_pretrain_team_kda/final_checkpoint/config.json (deflated 52%)
  adding: decoder_pretrain_team_kda/final_checkpoint/generation_config.json (deflated 39%)
  adding: decoder_pretrain_team_kda/final_checkpoint/tokenizer.json (deflated 84%)


In [ ]:
import os
from IPython.display import FileLink, display

os.chdir("/kaggle/working")

filename = "kda_team_decoder_final_checkpoint.zip"

print("Файл существует:", os.path.exists(filename))

if os.path.exists(filename):
    print("Размер:", os.path.getsize(filename), "байт")
    display(FileLink(filename))

Файл существует: True
Размер: 45644381 байт


/kaggle/working/kda_team_decoder_final_checkpoint.zip